In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
# Sample text dataset (just simple short strings)
texts = ["hi", "hello", "hey", "hola", "ciao"]
# Targets are just arbitrary numbers associated with each string (regression)
targets = torch.tensor([1.0, 2.0, 1.5, 2.5, 3.0])

In [5]:
# Build a simple character-to-index map
chars = sorted(list(set("".join(texts))))
char2idx = {c: i for i, c in enumerate(chars)}

In [7]:
# Encode texts as fixed-length vectors by one-hot encoding each character and flattening
def encode(text, max_len=5):
    vec = torch.zeros(max_len, len(chars))
    for i, c in enumerate(text):
        if i >= max_len:
            break
        vec[i, char2idx[c]] = 1.0
    return vec.view(-1)  # flatten

X = torch.stack([encode(t) for t in texts])

In [8]:
# Simple linear model: input size = max_len * number of chars, output size = 1
class LinearTextModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)

model = LinearTextModel(X.shape[1])

In [9]:
# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [10]:
# Training loop
for epoch in range(100):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, targets)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 20, Loss: 0.0447
Epoch 40, Loss: 0.0077
Epoch 60, Loss: 0.0014
Epoch 80, Loss: 0.0003
Epoch 100, Loss: 0.0000


In [11]:
# Test prediction on new input
test_text = "hey"
test_input = encode(test_text)
pred = model(test_input.unsqueeze(0))
print(f"Prediction for '{test_text}': {pred.item():.4f}")

Prediction for 'hey': 1.4936
